In [1]:
# ============================================
#  - checkpoints / checkpoints_mixed 기준 자동 로드
#  - YOLO + CLIP + H1/H2/H3/H4
#  - 그룹 색상 요약 + 중복 코디 금지 LLM 프롬프트
#  - ipywidgets FileUpload + 버튼으로 실시간 시연
# ============================================

import os, sys, json, math, warnings, io, traceback, re, collections
from pathlib import Path
from typing import List, Optional, Dict, Any
from dataclasses import dataclass, asdict

import platform
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import open_clip
from PIL import Image

# YOLO
try:
    from ultralytics import YOLO
    YOLO_AVAILABLE = True
except Exception:
    YOLO_AVAILABLE = False
    print("[Warning] 'ultralytics' import 실패 → YOLO 기능 비활성화.")

# LLM / Pydantic
from pydantic import BaseModel, Field
from openai import OpenAI
from getpass import getpass

# 위젯
import ipywidgets as widgets
from IPython.display import display, clear_output

# ------------------------------
# 0. 환경 정보 / 프로젝트 루트 설정
# ------------------------------
warnings.filterwarnings("ignore", category=UserWarning)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available(), "| python:", platform.python_version())

PROJECT_ROOT = Path().resolve()
os.chdir(PROJECT_ROOT)
print("CWD:", os.getcwd())
print("여기 있는 항목들:")
for p in PROJECT_ROOT.iterdir():
    print(" -", p)

CKPT_DIR = PROJECT_ROOT / "checkpoints"
MIXED_CKPT_DIR = PROJECT_ROOT / "checkpoints_mixed"
print("\ncheckpoints 상대경로 존재?", CKPT_DIR.exists(), "| 경로:", CKPT_DIR.resolve())
print("checkpoints_mixed 상대경로 존재?", MIXED_CKPT_DIR.exists(), "| 경로:", MIXED_CKPT_DIR.resolve())

USE_CUDA = torch.cuda.is_available()
DEVICE = "cuda" if USE_CUDA else "cpu"
VISION_DEVICE = DEVICE
print(f"\n--- DEVICE 설정: {DEVICE} ---")

# ------------------------------
# 1. 레이블 / 후보 리스트 정의 (13종 패션 라벨 고정)
# ------------------------------
MODEL_NAME = "ViT-B-32"
PRETRAINED_SOURCE = "laion2b_s34b_b79k"

MATERIALS = ["denim", "leather", "cotton", "wool", "silk", "knit", "suede", "fur"]
COLORS    = ["black", "white", "gray", "red", "blue", "green", "yellow", "pink", "purple", "orange", "brown", "beige"]

YOLO_LABELS = [
    "short sleeve top", "long sleeve top", "short sleeve outwear",
    "long sleeve outwear", "vest", "sling", "shorts",
    "trousers", "skirt", "short sleeve dress",
    "long sleeve dress", "vest dress", "sling dress",
]
H3_LABELS = YOLO_LABELS.copy()
N_H3_CLASSES = len(H3_LABELS)

print(f"\n--- 사용 YOLO/H3 레이블: {len(YOLO_LABELS)}종 (예: {YOLO_LABELS[0]}) ---")

# ------------------------------
# 2. 체크포인트 유틸 함수
# ------------------------------
def find_weight(patterns: List[str], start: Path) -> Optional[Path]:
    if not start.exists():
        return None
    for pat in patterns:
        hits = sorted(start.glob(pat))
        if hits:
            return hits[0]
    return None

def _extract_state_dict(obj: dict) -> dict:
    if isinstance(obj, dict):
        for k in ["state_dict", "model", "net", "module"]:
            if k in obj and isinstance(obj[k], dict):
                return _extract_state_dict(obj[k])
    return obj if isinstance(obj, dict) else {}

def _strip_prefix_keys(sd: dict, prefix: str) -> dict:
    out = {}
    for k, v in sd.items():
        if k.startswith(prefix):
            out[k[len(prefix):]] = v
    return out

def _infer_h4_head_spec(cls_sd: dict, emb_dim: int):
    use_layernorm = False
    hidden_dim = 256
    out_dim = 2
    w0 = cls_sd.get("0.weight", None)
    w3 = cls_sd.get("3.weight", None)
    if w0 is not None:
        if getattr(w0, "ndim", None) == 1:
            use_layernorm = True
            for idx in [1, 2, 3, 4]:
                w_lin = cls_sd.get(f"{idx}.weight", None)
                if (
                    w_lin is not None
                    and getattr(w_lin, "ndim", None) == 2
                    and w_lin.shape[1] == emb_dim
                ):
                    hidden_dim = w_lin.shape[0]
                    break
        elif getattr(w0, "ndim", None) == 2 and w0.shape[1] == emb_dim:
            hidden_dim = w0.shape[0]
    if w3 is not None and getattr(w3, "ndim", None) == 2:
        out_dim = w3.shape[0]
    return use_layernorm, hidden_dim, out_dim

def _build_h4_head_from_spec(emb_dim: int, use_layernorm: bool, hidden_dim: int, out_dim: int):
    layers = []
    if use_layernorm:
        layers += [
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5),
        ]
    else:
        layers += [
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5),
        ]
    layers += [nn.Linear(hidden_dim, out_dim)]
    return nn.Sequential(*layers)

def _safe_load_head(module: nn.Module, sd_full: dict, prefix: str = ""):
    sd = _extract_state_dict(sd_full)
    if prefix:
        sd = _strip_prefix_keys(sd, prefix)
    cur = module.state_dict()
    compat = {k: v for k, v in sd.items() if k in cur and cur[k].shape == v.shape}
    module.load_state_dict(compat, strict=False)
    skipped = [k for k in sd if k not in compat]
    if skipped:
        print(f"[SAFE-LOAD] skipped {len(skipped)} keys for {module.__class__.__name__} (shape mismatch).")

def _safe_mkdir(path: str):
    Path(path).mkdir(parents=True, exist_ok=True)

# ------------------------------
# 3. OpenCLIP 로드 (vision encoder 공유)
# ------------------------------
print("\nLoading OpenCLIP base model (시간 다소 소요)...")
clip_model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME, pretrained=PRETRAINED_SOURCE, device=VISION_DEVICE
)
clip_model = clip_model.eval()
EMB_DIM = clip_model.visual.output_dim
print(f"Base model loaded. EMB_DIM={EMB_DIM}")

# ------------------------------
# 4. 헤드 클래스 정의 (H1/H2/H3/H4)
# ------------------------------
class H4ContextHead(nn.Module):
    def __init__(self, emb_dim=EMB_DIM, classifier_head: Optional[nn.Module] = None):
        super().__init__()
        self.clip_vision_model = clip_model.visual
        for p in self.clip_vision_model.parameters():
       	    p.requires_grad = False
        self.classifier_head = classifier_head or nn.Sequential(
            nn.Linear(emb_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 2),
        )

    def forward(self, images):
        return self.classifier_head(self.clip_vision_model(images))

class H1AestheticHead(nn.Module):
    def __init__(self, emb_dim=EMB_DIM):
        super().__init__()
        self.clip_vision_model = clip_model.visual
        for p in self.clip_vision_model.parameters():
            p.requires_grad = False
        D = emb_dim
        self.aesthetic_head = nn.Sequential(
            nn.LayerNorm(D),
            nn.Linear(D, D // 2),
            nn.GELU(),
            nn.Linear(D // 2, 1),
        )

    def forward(self, images):
        f = self.clip_vision_model(images)
        output = self.aesthetic_head(f).squeeze(1)
        # 1~10 스케일
        return torch.sigmoid(output) * 9.0 + 1.0

class H3ItemClassifier(nn.Module):
    def __init__(self, emb_dim=EMB_DIM, num_classes=N_H3_CLASSES):
        super().__init__()
        self.clip_vision_model = clip_model.visual
        for p in self.clip_vision_model.parameters():
            p.requires_grad = False
        self.classifier_head = nn.Sequential(
            nn.Linear(emb_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, images):
        return self.classifier_head(self.clip_vision_model(images))

class H2OutfitHead(nn.Module):
    def __init__(self, emb_dim=EMB_DIM):
        super().__init__()
        D = emb_dim
        P = 128
        self.refiner_head = nn.Sequential(
            nn.Linear(D, D),
            nn.ReLU(),
            nn.Linear(D, P),
        )

    def forward(self, raw_clip_features):
        return self.refiner_head(raw_clip_features)

# ------------------------------
# 5. 헤드 인스턴스 + 가중치 로드
# ------------------------------
print("\n--- H1/H2/H3/H4 가중치 로드 ---")

# H4
h4_path = find_weight(["h4_best_*.pt", "h4*.pt", "h4*.pth"], start=CKPT_DIR)
if not h4_path or not h4_path.exists():
    raise FileNotFoundError(f"H4 weights not found under {CKPT_DIR}")
raw = torch.load(h4_path, map_location="cpu")
sd_full = _extract_state_dict(raw)
sd_cls = _strip_prefix_keys(sd_full, "classifier_head.")
use_ln, hid_dim, out_dim = _infer_h4_head_spec(sd_cls, EMB_DIM)
print(f"[H4] inferred spec: LayerNorm={use_ln}, hidden_dim={hid_dim}, out_dim={out_dim}")
h4_head = _build_h4_head_from_spec(EMB_DIM, use_ln, hid_dim, out_dim)
h4_model = H4ContextHead(classifier_head=h4_head).to(VISION_DEVICE).eval()
_safe_load_head(h4_model.classifier_head, sd_full, "classifier_head.")
print(f"Trained H4 loaded: {h4_path}")

# H1
H1_MODEL = H1AestheticHead().to(VISION_DEVICE).eval()
h1_path = find_weight(["h1_best_*.pt", "h1*.pt", "h1*.pth"], start=CKPT_DIR)
if h1_path and h1_path.exists():
    st = torch.load(h1_path, map_location="cpu")
    _safe_load_head(H1_MODEL.aesthetic_head, st, "head.")
    print(f"Trained H1 loaded: {h1_path}")
else:
    H1_MODEL = None
    print("[Warning] H1 weights not found.")

# H3 (새 mixed 모델)
H3_MODEL = H3ItemClassifier(num_classes=N_H3_CLASSES).to(VISION_DEVICE).eval()
h3_path = find_weight(["h3_mixed_best_*.pt"], start=MIXED_CKPT_DIR)
if h3_path and h3_path.exists():
    st = torch.load(h3_path, map_location="cpu")
    _safe_load_head(H3_MODEL.classifier_head, st, "")
    print(f"Trained H3 (Mixed) loaded: {h3_path}")
else:
    H3_MODEL = None
    print("[Warning] H3 (Mixed) weights not found.")

# H2
H2_MODEL = H2OutfitHead().to(VISION_DEVICE).eval()
h2_path = find_weight(["h2_best_*.pt", "h2*.pt", "h2*.pth"], start=CKPT_DIR)
if h2_path and h2_path.exists():
    st = torch.load(h2_path, map_location="cpu")
    _safe_load_head(H2_MODEL.refiner_head, st, "projection_head.")
    print(f"Trained H2 loaded: {h2_path}")
else:
    H2_MODEL = None
    print("[Warning] H2 weights not found.")

print(f"\nAll models are ready on: DEVICE={DEVICE}")

# ------------------------------
# 6. LLM 모듈 정의 (중복 코디 금지 + group_colors)
# ------------------------------
print("\n--- LLM 모듈 설정 ---")

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
if not api_key or api_key.startswith("여기에_"):
    print("OpenAI API 키가 필요합니다. (sk-... 또는 sk-proj-...)")
    try:
        api_key = getpass("OpenAI API Key: ").strip()
    except Exception:
        api_key = input("OpenAI API Key: ").strip()

if api_key:
    os.environ["OPENAI_API_KEY"] = api_key
    print("OpenAI API Key 설정 완료.")
else:
    print("[경고] OPENAI_API_KEY가 설정되지 않았습니다. LLM 호출 시 오류가 날 수 있습니다.")

class AnalysisInput(BaseModel):
    context: str
    main_material: str
    main_color: str
    detected_items: List[str]
    group_colors: Dict[str, Any] = {}

class FashionAdvice(BaseModel):
    one_line_summary: str = Field(description="한 문장으로 된, 긍정적인 스타일 총평")
    positive_points: List[str] = Field(description="이 스타일의 매력적인 부분이나 칭찬할 만한 점 2-3가지")
    suggestion: str = Field(description="스타일을 한 단계 더 업그레이드할 수 있는 구체적이고 실천 가능한 아이템 또는 팁 1가지")

SYSTEM_PROMPT = """
당신은 사용자의 자신감을 북돋우는 전문 AI 스타일리스트입니다.
[목표]
- 입력으로 주어진 4가지 정보(맥락, 주요 재질, 주요 색, 탐지된 아이템 리스트)와 '그룹별 색상 요약(group_colors)'을 바탕으로,
  깔끔하고 실용적인 코디 피드백을 생성한다.
- 출력은 아래 JSON 템플릿의 정확한 키만 포함해야 한다.
[톤 & 문체]
- 한국어, 반말과 존댓말 사이의 중립적이고 공손한 서술체.
- 평가는 긍정 먼저 → 개선 제안 순서.
[색상/품목 매핑 절대 규칙]
- 상의 관련 설명은 group_colors.tops.primary를, 하의 관련 설명은 group_colors.bottoms.primary를 우선 사용한다.
- 드레스가 있을 경우 group_colors.dresses.primary를 우선 설명하고 상·하 분리는 하지 않는다.
- tops/bottoms/dresses 팔레트에 없는 색을 새로 만들지 않는다. (환각 금지)
[중복 코디 금지 규칙 — 매우 중요]
- '현재 착장'과 동일한 조합을 권하지 말라. (예: 현재가 노란 상의 + 블루 하의면 동일 조합 추천 금지)
- 제안은 다음 중 정확히 하나만 수행: (a) 상의 또는 하의 한 파츠만 교체, 또는 (b) 액세서리/아우터/신발 1가지를 추가.
- (a)라면 바꿀 파츠의 구체 색/소재/실루엣을 명시하고, 근거를 1문장으로 쓴다.
- (b)라면 아이템 종류, 구체 색, 매칭 근거(보색/무채색/톤온톤 등)를 1문장으로 쓴다.
- detected_items의 아이템명(예: 'trousers','long sleeve top')을 우리에게 친숙한 아이템명으로(예: 'trousers = 바지', 'short sleeve top = 반팔') 최소 1개를 suggestion에 명시한다.
[TPO 가이드]
- consumer: 활동성/세탁 난이도/실용성 우선.
- shop: 실루엣 대비/질감 대비/시선 포인트/트렌드 키워드 강조.
[재질·색 조합 간단 룰셋]
- denim: 화이트/그레이/네이비 안정.
- leather: 광택 강하면 다른 파츠는 매트로 밸런스.
- knit/wool: 부피감↑ → 하의는 직선 실루엣로.
- silk: 드레이프·광택 강조, 하의는 표면 거친 소재와 대비.
[출력 형식]
- 스키마 외 키 추가 금지, 줄바꿈/주석 금지.
"""

def _safe_parse_json(s: str) -> dict:
    m = re.search(r"\{.*\}", s, flags=re.S)
    if m:
        s = m.group(0)
    return json.loads(s)

def _group_items_from_names(names: List[str]) -> Dict[str, List[str]]:
    tops, bottoms, dresses = [], [], []
    for n in names:
        n_low = (n or "").lower()
        if "dress" in n_low:
            dresses.append(n)
        elif n_low in {"trousers", "shorts", "skirt"}:
            bottoms.append(n)
        elif ("top" in n_low) or ("outwear" in n_low) or (n_low in {"vest", "sling"}):
            tops.append(n)
    return {"tops": tops, "bottoms": bottoms, "dresses": dresses}

def _violates_novelty(text: str, tops_primary: Optional[str], bottoms_primary: Optional[str], grp_items: Dict[str, List[str]]) -> bool:
    t = (text or "").lower()
    tp = (tops_primary or "").lower()
    bp = (bottoms_primary or "").lower()
    has_top_item = any(
        ("top" in i.lower()) or ("outwear" in i.lower()) or (i.lower() in {"vest", "sling"})
        for i in grp_items.get("tops", [])
    )
    has_bottom_item = any(i.lower() in {"trousers", "shorts", "skirt"} for i in grp_items.get("bottoms", []))
    return bool(tp and bp and has_top_item and has_bottom_item and (tp in t) and (bp in t))

def advise_with_chatgpt(ai_input: AnalysisInput) -> FashionAdvice:
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

    items_str = ", ".join(ai_input.detected_items) if ai_input.detected_items else "탐지된 아이템 없음"
    gc = ai_input.group_colors or {}
    tops_primary    = (gc.get("tops", {}) or {}).get("primary")
    bottoms_primary = (gc.get("bottoms", {}) or {}).get("primary")
    dresses_primary = (gc.get("dresses", {}) or {}).get("primary")

    grp_items = _group_items_from_names(ai_input.detected_items)
    snapshot = {
        "tops":    {"items": grp_items["tops"],    "primary": tops_primary},
        "bottoms": {"items": grp_items["bottoms"], "primary": bottoms_primary},
        "dresses": {"items": grp_items["dresses"], "primary": dresses_primary},
    }
    snapshot_json = json.dumps(snapshot, ensure_ascii=False)
    group_colors_json = json.dumps(ai_input.group_colors, ensure_ascii=False)

    def _one_call(extra_user_note: str = "") -> FashionAdvice:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content":
                    f"맥락: {ai_input.context}\n"
                    f"주요 재질: {ai_input.main_material}\n"
                    f"전신 대표색: {ai_input.main_color}\n"
                    f"탐지된 아이템: [{items_str}]\n"
                    f"그룹별 색상 요약(JSON): {group_colors_json}\n"
                    f"현재 착장 스냅샷(JSON): {snapshot_json}\n"
                    + (extra_user_note or "")
                    + "\n아래 템플릿과 동일한 키만 포함한 JSON으로만 답하세요:"
                    '\n{"one_line_summary":"","positive_points":[],"suggestion":""}'
            },
        ]

        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0.2,
            response_format={"type": "json_object"},
        )
        data = _safe_parse_json(completion.choices[0].message.content)
        return FashionAdvice.model_validate(data)

    advice = _one_call()
    combined_text = " ".join([advice.one_line_summary, *advice.positive_points, advice.suggestion])
    if _violates_novelty(combined_text, tops_primary, bottoms_primary, grp_items):
        # 동일 상·하 조합을 추천한 경우 재요청
        advice = _one_call(
            extra_user_note=(
                "\n[중복 금지 강화] 방금 응답은 현재 착장을 그대로 권했습니다. "
                "같은 상·하 조합을 반복하지 말고, "
                "(a) 상의 또는 하의 한 파츠만 교체하거나 "
                "(b) 액세서리/아우터/신발 1가지를 추가하는 방식으로 제안하세요."
            )
        )
    return advice

print("LLM 모듈 준비 완료.")

# ------------------------------
# 7. Vision 파이프라인 헬퍼 정의
# ------------------------------
@dataclass
class DetectedItem:
    category: str
    bbox: List[float]
    score: float
    h3_category: Optional[str] = None
    crop_path: Optional[str] = None
    refined_feature_h2: Optional[torch.Tensor] = None
    colors_topk: Optional[List[Dict[str, float]]] = None

@dataclass
class AnalysisSummary:
    context_pred: str
    context_probs: Dict[str, float]
    main_material: str
    main_color: str
    items: List[DetectedItem]
    aesthetics_score_h1: Optional[float]
    compatibility_score_h2: Optional[float]
    main_colors_topk: Optional[List[Dict[str, float]]] = None
    main_materials_topk: Optional[List[Dict[str, float]]] = None
    group_colors: Optional[Dict[str, Any]] = None

@torch.inference_mode()
def extract_feature(image: Image.Image, text_candidates: List[str]) -> str:
    image_input = preprocess(image).unsqueeze(0).to(DEVICE)
    text_inputs = open_clip.tokenize(text_candidates).to(DEVICE)
    img_f = clip_model.encode_image(image_input)
    txt_f = clip_model.encode_text(text_inputs)
    img_f = img_f / img_f.norm(dim=-1, keepdim=True)
    txt_f = txt_f / txt_f.norm(dim=-1, keepdim=True)
    sim = (100.0 * img_f @ txt_f.T).softmax(dim=-1)
    return text_candidates[int(sim.argmax().item())]

@torch.inference_mode()
def extract_topk_features(image: Image.Image, text_candidates: List[str], top_k: int = 3, min_prob: float = 0.12):
    image_input = preprocess(image).unsqueeze(0).to(DEVICE)
    text_inputs = open_clip.tokenize(text_candidates).to(DEVICE)
    img_f = clip_model.encode_image(image_input)
    txt_f = clip_model.encode_text(text_inputs)
    img_f = img_f / img_f.norm(dim=-1, keepdim=True)
    txt_f = txt_f / txt_f.norm(dim=-1, keepdim=True)
    sim = (100.0 * img_f @ txt_f.T).softmax(dim=-1).squeeze(0)
    probs = sim.detach().float().cpu().tolist()
    ranked = sorted(list(zip(text_candidates, probs)), key=lambda x: x[1], reverse=True)
    picked = []
    for label, p in ranked[: max(top_k, 1)]:
        if p >= min_prob:
            picked.append((label, float(p)))
    return picked

@torch.inference_mode()
def _infer_context_with_h4(image: Image.Image) -> Dict[str, Any]:
    H4_CLASS_NAMES = ["shop", "consumer"]
    x = preprocess(image).unsqueeze(0).to(DEVICE)
    logits = h4_model(x)
    if logits.shape[-1] == 1:
        p1 = torch.sigmoid(logits).item()
        probs = [1 - p1, p1]
    else:
        probs = F.softmax(logits, dim=-1).squeeze(0).tolist()
    idx = int(torch.tensor(probs, device=DEVICE).argmax().item())
    return {
        "context": H4_CLASS_NAMES[idx],
        "probs": {H4_CLASS_NAMES[i]: float(p) for i, p in enumerate(probs)},
    }

def _detect_yolo(image_path: str, save_dir: str) -> List[DetectedItem]:
    """YOLO 크롭을 PNG로 저장하여 Pillow JPEG 관련 이슈 회피."""
    items: List[DetectedItem] = []
    if not YOLO_AVAILABLE:
        print("YOLO not available, skipping detection.")
        return items
    yolo_weights = MIXED_CKPT_DIR / "yolo_mixed_run" / "weights" / "best.pt"
    if not yolo_weights.exists():
        print(f"YOLO (Mixed) weights not found at: {yolo_weights}")
        raise FileNotFoundError(str(yolo_weights))

    print(f"--- Using YOLO model: {yolo_weights} ---")
    model = YOLO(yolo_weights)
    device_arg = 0 if USE_CUDA else "cpu"
    res = model.predict(
        image_path,
        device=device_arg,
        imgsz=896,
        conf=0.25,
        iou=0.5,
        half=False,
        verbose=False,
    )

    img = Image.open(image_path).convert("RGB")
    _safe_mkdir(save_dir)
    img_basename = Path(image_path).stem

    for r in res:
        for b in r.boxes:
            cls = int(b.cls.item())
            score = float(b.conf.item())
            x1, y1, x2, y2 = map(float, b.xyxy[0].tolist())
            crop = img.crop((x1, y1, x2, y2)).convert("RGB")
            crop_name = f"{img_basename}_crop_{cls}_{int(x1)}_{int(y1)}.png"
            crop_path = str(Path(save_dir) / crop_name)
            crop.save(crop_path, format="PNG")

            items.append(
                DetectedItem(
                    category=YOLO_LABELS[cls] if cls < len(YOLO_LABELS) else f"cls_{cls}",
                    bbox=[x1, y1, x2, y2],
                    score=score,
                    crop_path=crop_path,
                )
            )

    print(f"YOLO detected {len(items)} items ({'CUDA' if USE_CUDA else 'CPU'}).")
    return items

@torch.inference_mode()
def _classify_items_h3_and_refine_h2(items: List[DetectedItem]) -> None:
    if not items:
        return
    print(f"Running H3 classification & H2 refinement for {len(items)} items...")
    for it in items:
        if not it.crop_path:
            continue
        try:
            crop_img = Image.open(it.crop_path).convert("RGB")
            x = preprocess(crop_img).unsqueeze(0).to(DEVICE)

            # H3
            if H3_MODEL:
                logits = H3_MODEL(x)
                pred = int(logits.argmax().item())
                it.h3_category = H3_LABELS[pred]

            # H2
            if H2_MODEL:
                raw_clip_feature = clip_model.encode_image(x)
                it.refined_feature_h2 = H2_MODEL(raw_clip_feature)

            # 아이템별 색 Top-K (상위 2개, 0.15 이상만)
            col_topk = extract_topk_features(crop_img, COLORS, top_k=2, min_prob=0.15)
            it.colors_topk = [{"label": l, "prob": round(p, 4)} for (l, p) in col_topk]
        except Exception as e:
            print(f"[H3/H2] error on {it.crop_path}: {e}")

def _summarize_group_colors(items: List[DetectedItem]) -> Dict[str, Any]:
    """탑/바텀/드레스 그룹별 팔레트 및 대표색(primary) 산출."""
    groups = {"tops": [], "bottoms": [], "dresses": []}
    for it in items:
        cat = (it.h3_category or it.category or "").strip()
        if not it.colors_topk:
            continue
        pal = [
            (d["label"], float(d["prob"]))
            for d in it.colors_topk
            if "label" in d and "prob" in d
        ]
        if not pal:
            continue

        if "dress" in cat:
            groups["dresses"].extend(pal)
        elif cat in {"trousers", "shorts", "skirt"}:
            groups["bottoms"].extend(pal)
        elif ("top" in cat) or ("outwear" in cat) or (cat in {"vest", "sling"}):
            groups["tops"].extend(pal)
        else:
            groups["tops"].extend(pal)

    out = {}
    for key in ["tops", "bottoms", "dresses"]:
        pal = groups[key]
        acc = collections.defaultdict(float)
        for label, w in pal:
            acc[label] += w
        ranked = sorted(acc.items(), key=lambda kv: kv[1], reverse=True)
        palette_top3 = [{"label": l, "weight": round(w, 4)} for (l, w) in ranked[:3]]
        primary = ranked[0][0] if ranked else None
        out[key] = {"primary": primary, "palette": palette_top3}
    return out

@torch.inference_mode()
def _score_aesthetic_h1(image: Image.Image) -> Optional[float]:
    if H1_MODEL is None:
        print("H1 model not loaded...")
        return None
    try:
        x = preprocess(image).unsqueeze(0).to(DEVICE)
        score_1_to_10 = H1_MODEL(x)
        return float(score_1_to_10.squeeze().item())
    except Exception as e:
        print(f"[H1] error: {e}")
        return None

@torch.inference_mode()
def _compute_compatibility_score_h2(items: List[DetectedItem]) -> Optional[float]:
    if H2_MODEL is None:
        print("H2 model not loaded...")
        return None
    item_embeddings = [it.refined_feature_h2 for it in items if it.refined_feature_h2 is not None]
    if len(item_embeddings) < 2:
        print(f"H2 needs at least 2 items, found {len(item_embeddings)}. Skipping score.")
        return None
    pair_scores = []
    for i in range(len(item_embeddings)):
        for j in range(i + 1, len(item_embeddings)):
            sim = torch.nn.functional.cosine_similarity(item_embeddings[i], item_embeddings[j]).item()
            pair_scores.append(sim)
    if not pair_scores:
        return None
    avg_similarity = sum(pair_scores) / len(pair_scores)
    score_1_to_10 = (avg_similarity + 1.0) * 4.5 + 1.0
    return float(score_1_to_10)

# ------------------------------
# 8. E2E 메인 파이프라인 함수
# ------------------------------
def analyze_image_end2end(image_path: str, workdir: str = "./_e2e_runs") -> Dict[str, Any]:
    print("=" * 50)
    print(f"--- 1. Analyzing Image: ...{image_path[-40:]} ---")

    _safe_mkdir(workdir)
    try:
        img = Image.open(image_path).convert("RGB")
    except Exception as e:
        print(f"[Fatal Error] Failed to open image: {e}")
        return {"error": f"Failed to open image: {e}"}

    # 1) YOLO
    print("--- 2. Running YOLOv8 Detection ---")
    img_basename = os.path.splitext(os.path.basename(image_path))[0]
    yolo_dir = os.path.join(workdir, f"{img_basename}_crops")
    items = _detect_yolo(image_path, yolo_dir)

    # 2) CLIP 재질/색 (전신)
    print("--- 3. Running CLIP Feature Extraction (Material/Color) ---")
    main_material = extract_feature(img, MATERIALS)
    main_color = extract_feature(img, COLORS)
    mat_topk = extract_topk_features(img, MATERIALS, top_k=3, min_prob=0.12)
    col_topk = extract_topk_features(img, COLORS, top_k=3, min_prob=0.12)
    mat_topk_json = [{"label": l, "prob": round(p, 4)} for (l, p) in mat_topk]
    col_topk_json = [{"label": l, "prob": round(p, 4)} for (l, p) in col_topk]

    # 3) H4 컨텍스트
    print("--- 4. Running H4 Context Head ---")
    h4_result = _infer_context_with_h4(img)
    context = h4_result["context"]

    # 4) H3/H2 + 아이템별 색
    print("--- 5. Running H3 (Classification) & H2 (Refinement) ---")
    _classify_items_h3_and_refine_h2(items)

    # 5) 그룹별 색상 요약
    group_colors = _summarize_group_colors(items)

    # 6) H1 심미성 점수
    print("--- 6. Running H1 (Aesthetic Score) ---")
    h1_score = _score_aesthetic_h1(img)

    # 7) H2 호환성 점수
    print("--- 7. Running H2 (Compatibility Score) ---")
    h2_score = _compute_compatibility_score_h2(items)

    # H2 텐서 삭제 (JSON 직렬화용)
    items_cleaned = []
    for it in items:
        items_cleaned.append(
            asdict(it, dict_factory=lambda x: {k: v for (k, v) in x if k != "refined_feature_h2"})
        )

    summary = AnalysisSummary(
        context_pred=context,
        context_probs=h4_result["probs"],
        main_material=main_material,
        main_color=main_color,
        items=items_cleaned,
        aesthetics_score_h1=h1_score,
        compatibility_score_h2=h2_score,
        main_colors_topk=col_topk_json,
        main_materials_topk=mat_topk_json,
        group_colors=group_colors,
    )

    # 8) LLM 조언
    print("--- 8. Generating final advice with ChatGPT... ---")
    try:
        item_names = [(it.get("h3_category") or it.get("category")) for it in items_cleaned]
        item_names_unique = sorted(list(set(item_names)))
        ai_input = AnalysisInput(
            context=context,
            main_material=main_material,
            main_color=main_color,
            detected_items=item_names_unique,
            group_colors=group_colors,
        )
        advice = advise_with_chatgpt(ai_input)
        advice_dict = advice.model_dump()
    except Exception as e:
        print(f"[Error] LLM advice generation failed: {e}")
        advice_dict = {"error": str(e)}

    result = {
        "inputs": {"image_path": image_path},
        "analysis": asdict(summary),
        "fashion_advice": advice_dict,
    }

    print("=" * 50)
    print("--- E2E Analysis Function Defined (Grouped Color Mapping + Novelty) ---")
    return result

print("\nE2E 함수 'analyze_image_end2end' 정의 완료.")

# ------------------------------
# 9. ipywidgets 기반 실시간 데모 (WSL/Jupyter)
# ------------------------------
print("\n--- 실시간 데모 위젯 로드 중... ---")

uploader = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="1) 이미지 업로드",
)
run_btn = widgets.Button(
    description="2) 분석 실행",
    button_style="primary",
)
out_image = widgets.Output()
out_json = widgets.Output()
status = widgets.Output()

def _extract_from_uploader(val):
    """FileUpload 위젯 value에서 (이름, bytes) 튀어나오게 하는 유틸."""
    if not val:
        return None, None
    # JupyterLab/Notebook 환경마다 구조가 달라서 방어적으로 처리
    if isinstance(val, dict):
        fname = next(iter(val.keys()), None)
        if not fname:
            return None, None
        info = val[fname]
        data = info.get("content", None)
        return fname, data
    if isinstance(val, (tuple, list)) and len(val) > 0:
        it = val[0]
        if hasattr(it, "content"):
            return getattr(it, "name", "uploaded_image"), it.content
        if isinstance(it, dict):
            if "content" in it:
                return it.get("name", "uploaded_image"), it["content"]
            k = next(iter(it.keys()), None)
            sub = it.get(k, {}) if k else {}
            return k, sub.get("content")
    return None, None

def _reset_uploader():
    try:
        uploader.value.clear()
    except Exception:
        try:
            uploader.value = ()
        except Exception:
            uploader.value = {}

def _analyze_bytes(name, data_bytes):
    out_image.clear_output()
    out_json.clear_output()
    status.clear_output()
    try:
        with out_image:
            print(f"업로드: {name} ({len(data_bytes)} bytes)")
            display(Image.open(io.BytesIO(data_bytes)))
            print("--- 분석 시작 (GPU) ---")

        temp_dir = "./_e2e_runs"
        os.makedirs(temp_dir, exist_ok=True)
        temp_path = os.path.join(temp_dir, f"live_demo_{name}")
        with open(temp_path, "wb") as f:
            f.write(data_bytes)

        result = analyze_image_end2end(temp_path, workdir=temp_dir)

        with out_json:
            print("최종 JSON")
            print(json.dumps(result, ensure_ascii=False, indent=2))

            save_path = Path(temp_dir) / f"{Path(temp_path).stem}_result.json"
            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(result, f, ensure_ascii=False, indent=2)
            print("JSON 저장:", save_path.resolve())
    except Exception as e:
        with out_json:
            print(f"[Fatal] {e}")
            traceback.print_exc()

def on_run_clicked(_):
    status.clear_output()
    with status:
        print("업로더 버퍼에서 파일 읽는 중...")
    name, data = _extract_from_uploader(uploader.value)
    if not name or data is None:
        with status:
            print("업로더에 파일이 없습니다. 먼저 파일을 올려주세요.")
        return
    _analyze_bytes(name, data)
    _reset_uploader()

run_btn.on_click(on_run_clicked)

display(
    widgets.VBox(
        [
            widgets.HBox([uploader, run_btn]),
            status,
            out_image,
            out_json,
        ]
    )
)
print("'1) 이미지 업로드'로 파일을 올린 뒤 '2) 분석 실행' 버튼을 누르면 됩니다.")
print("\nWSL/Jupyter용 실시간 패션 분석 데모 준비 완료.")


torch: 2.8.0+cu128
cuda available: True | python: 3.11.14
CWD: /home/epistachio/projects/fashion_ai
여기 있는 항목들:
 - /home/epistachio/projects/fashion_ai/h4_context_head_epoch_3.pth
 - /home/epistachio/projects/fashion_ai/fashion_ai_backend.zip
 - /home/epistachio/projects/fashion_ai/yolo11n.pt
 - /home/epistachio/projects/fashion_ai/deepfashion2.yaml
 - /home/epistachio/projects/fashion_ai/checkpoints_mixed
 - /home/epistachio/projects/fashion_ai/__pycache__
 - /home/epistachio/projects/fashion_ai/GPU_availvale_test.ipynb
 - /home/epistachio/projects/fashion_ai/manifest.csv
 - /home/epistachio/projects/fashion_ai/requirements.txt
 - /home/epistachio/projects/fashion_ai/chat_api.py
 - /home/epistachio/projects/fashion_ai/H2_1.ipynb
 - /home/epistachio/projects/fashion_ai/Untitled.ipynb
 - /home/epistachio/projects/fashion_ai/yolov8n.pt
 - /home/epistachio/projects/fashion_ai/main.py
 - /home/epistachio/projects/fashion_ai/manifest_with_scores.csv
 - /home/epistachio/projects/fashion_ai/fa

OpenAI API Key:  ········


✅ OpenAI API Key 설정 완료.
✅ LLM 모듈 준비 완료.

✅ E2E 함수 'analyze_image_end2end' 정의 완료.

--- 실시간 데모 위젯 로드 중... ---


⏳ '1) 이미지 업로드'로 파일을 올린 뒤 '2) 분석 실행' 버튼을 누르면 됩니다.

✅ WSL/Jupyter용 실시간 패션 분석 데모 준비 완료.
